# TP2 NLP - Text-to-SQL Workflow (All-in-One)
Este notebook atua como ponto de entrada **único** para rodar a pipeline completa de fine-tuning QLoRA e a avaliação (Spider + MMLU) em um ambiente Google Colab com GPU.

In [ ]:
# Integração com o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuração do Ambiente e Repositório
!git clone https://github.com/mh131105/TP2_NLP.git
%cd TP2_NLP
!pip install -r requirements.txt
!python src/environment.py

In [ ]:
# Autenticação no Hugging Face (Login)
# Necessário para modelos restritos ou para salvar repositórios na sua conta
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Download antecipado do Hugging Face (Modelos e Datasets)
import os
from transformers import AutoTokenizer
from datasets import load_dataset

model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
print(f"Baixando tokenizador e cacheando modelo {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Baixando base MMLU...")
mmlu = load_dataset("cais/mmlu", "all", split="test")
print("Downloads do HF concluídos.")

In [ ]:
# Download do dataset Spider via Hugging Face Datasets (Contorna limite do GDrive)
import os
import shutil
from datasets import load_dataset
import datasets.config

if not os.path.exists('data/raw/spider/train_spider.json'):
    print('Baixando dataset Spider via Hugging Face (CDN)...')
    # A biblioteca datasets baixa o spider.zip e contorna os limites do GDrive via CDN interno
    ds = load_dataset('spider')
    
    print('Localizando os bancos de dados (.sqlite) e JSONs extraidos no cache...')
    extracted_dir = os.path.join(datasets.config.HF_DATASETS_CACHE, 'downloads', 'extracted')
    
    db_source_path = None
    for root, dirs, files in os.walk(extracted_dir):
        if 'database' in dirs and 'train_spider.json' in files:
            db_source_path = root
            break
            
    if db_source_path:
        print(f'Copiando arquivos brutos para data/raw/spider/')
        os.makedirs('data/raw', exist_ok=True)
        if os.path.exists('data/raw/spider'):
            shutil.rmtree('data/raw/spider')
        shutil.copytree(db_source_path, 'data/raw/spider')
        print('Dataset Spider extraído e posicionado com sucesso!')
    else:
        print('Erro: não foi possível encontrar os arquivos brutos do Spider no cache.')
else:
    print('Dataset Spider já encontrado localmente.')

In [ ]:
# Preparação dos dados (Spider e MMLU)
# Certifique-se de que os dados crus do Spider estão em data/raw/spider/
!python scripts/prepare_spider.py --config configs/data.yaml --split train
!python scripts/prepare_spider.py --config configs/data.yaml --split dev
!python scripts/prepare_mmlu.py --config configs/eval_mmlu.yaml

In [ ]:
# Avaliação Baseline no Spider Dev
!python scripts/run_baseline_spider.py --config configs/eval_spider.yaml

In [ ]:
# Fine-tuning QLoRA - Experimento A
!python scripts/train_qlora.py --config configs/train_exp_a.yaml

In [ ]:
# Avaliação do Experimento A no Spider Dev
!python scripts/evaluate_spider.py --config configs/eval_spider.yaml --adapter_path outputs/finetuned_exp_a/adapters

In [ ]:
# Fine-tuning QLoRA - Experimento B
!python scripts/train_qlora.py --config configs/train_exp_b.yaml

In [ ]:
# Avaliação do Experimento B no Spider Dev
!python scripts/evaluate_spider.py --config configs/eval_spider.yaml --adapter_path outputs/finetuned_exp_b/adapters

In [ ]:
# Avaliação MMLU (Regressão de Conhecimento)
# Baseline
!python scripts/evaluate_mmlu.py --config configs/eval_mmlu.yaml
# Experimento A
!python scripts/evaluate_mmlu.py --config configs/eval_mmlu.yaml --adapter_path outputs/finetuned_exp_a/adapters
# Experimento B
!python scripts/evaluate_mmlu.py --config configs/eval_mmlu.yaml --adapter_path outputs/finetuned_exp_b/adapters

In [ ]:
# Agregação de Resultados e Exportação de Tabelas
!python scripts/aggregate_results.py
!python scripts/export_report_tables.py

In [ ]:
# Salvar Artefatos Pesados e Benchmarks no Google Drive
import os

drive_path = "/content/drive/MyDrive/TP2_NLP_Outputs"
os.makedirs(drive_path, exist_ok=True)

print("Copiando Adapters (Experimento A e B) para o Drive...")
!cp -r outputs/finetuned_exp_a/adapters {drive_path}/adapters_exp_a
!cp -r outputs/finetuned_exp_b/adapters {drive_path}/adapters_exp_b

print("Copiando Métricas e Tabelas para o Drive...")
!cp -r outputs/metrics {drive_path}/metrics
!cp -r outputs/report_assets {drive_path}/report_assets

print("Backup concluído com sucesso no Google Drive!")